In [ ]:
import os
from pathlib import Path
import logging
import shutil
from datetime import datetime
import time

import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.evaluation import RemoteReadOnlyEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.usgs.usgs import usgs_to_parquet
from teehr.fetching.utils import create_periods_based_on_chunksize, get_period_start_end_times

logger = logging.getLogger()

- NWM v3.0: 2023-09-19 - present
- Earliest reference time in our secondary table: 2025-11-10 22:00:00

In [ ]:
CACHE_DIR = '/data/data_processing/backfill-usgs/usgs-teehr-warehouse/local/cache/fetching/usgs/usgs_observations/streamflow_hourly_inst'

#### Load to the warehouse

In [ ]:
%%time
# READ-WRITE
spark = create_spark_session(
    # start_spark_cluster=True,
    # executor_instances=4,
    # executor_memory="50g",
    # executor_cores=7,
    aws_profile="admin-user",
    update_configs={
        "spark.sql.shuffle.partitions": "52"
    }    
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
schema = ev.primary_timeseries.schema_func().to_structtype()

In [ ]:
options = {
    "header": "true",
    "ignoreMissingFiles": "true",
    "recursiveFileLookup": "true"
}
usgs_sdf = ev.spark.read.format("parquet").options(**options).load(CACHE_DIR, schema=schema)
usgs_sdf.show(n=4, truncate=False)

In [ ]:
usgs_sdf.select(F.min("value_time")).show(), usgs_sdf.select(F.max("value_time")).show()

In [ ]:
usgs_sdf.select("location_id").distinct().count()

In [ ]:
%%time
ev.table("primary_timeseries").load_dataframe(
    df=usgs_sdf,
    write_mode="append"
)

#### NOTE: If you are confident the data has been loaded to the warehouse delete the local evaluation data